<a href="https://colab.research.google.com/github/pantherer/DSA-Assignment/blob/main/ANN_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Python/Keras Code for 7088CEM Spam Classification Assignment ---

# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# --- 1. Data Formalization and Pre-processing (Section 3) ---

# Direct data file link from UCI Spambase dataset
UCI_DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"

# Load the data, instructing pandas not to look for headers
print("Loading Spambase dataset...")
data = pd.read_csv(UCI_DATA_URL, header=None)

# Data Cleaning: Remove duplicates
initial_rows = data.shape[0]
data.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - data.shape[0]} duplicate rows. Dataset size: {data.shape[0]}")

# Separate Features (X) and Target (Y). Features are columns 0-56 (57 total).
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values
INPUT_DIM = X.shape[1]

# Data Partitioning: 70% Train, 10% Validation, 20% Test (Section 3 Rationale)
# 1. Split off 20% for the final test set
X_train_val, X_test, Y_train_val, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, stratify=Y
)
# 2. Split the remaining 80% into 70% train and 10% validation (10/80 = 0.125)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_val, Y_train_val, test_size=0.125, random_state=42, stratify=Y_train_val
)
print(f"Data Split: Train={X_train.shape[0]}, Validation={X_val.shape[0]}, Test={X_test.shape[0]}")

# Feature Scaling: Min-Max Normalization (CRITICAL for ANN performance)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("Features Normalized using MinMaxScaler (0 to 1).")


# --- 2. Model Definition (Section 4) ---

# Function to create Model 1: Baseline MLP (Shallow)
def create_baseline_mlp(input_dim):
    # Justification: Simple network for fast training and benchmark establishment
    model = Sequential([
        # Hidden Layer: 32 Neurons, ReLU activation
        Dense(32, activation='relu', input_dim=input_dim),
        # Output Layer: 1 Neuron, Sigmoid activation (Binary Classification)
        Dense(1, activation='sigmoid')
    ])

    # Compile the model (Optimizer: Adam, Loss: Binary Cross-Entropy)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Function to create Model 2: Deep MLP (Advanced)
def create_deep_mlp(input_dim):
    # Justification: Deeper architecture to explore non-linear complexity
    model = Sequential([
        # Hidden Layer 1: 64 Neurons, ReLU activation
        Dense(64, activation='relu', input_dim=input_dim),
        # Hidden Layer 2: 32 Neurons, ReLU activation
        Dense(32, activation='relu'),
        # Output Layer: 1 Neuron, Sigmoid activation (Binary Classification)
        Dense(1, activation='sigmoid')
    ])

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model_baseline = create_baseline_mlp(INPUT_DIM)
print("\n--- Model 1: Baseline MLP Summary (for Appendix 1) ---")
model_baseline.summary()

model_deep = create_deep_mlp(INPUT_DIM)
print("\n--- Model 2: Deep MLP Summary (for Appendix 1) ---")
model_deep.summary()


# --- 3. Training and Evaluation (Section 5) ---

BATCH_SIZE = 32
EPOCHS = 50

# Optional: Adding Early Stopping for better generalization (Good for Discussion/Future Work)
# Monitors validation loss and stops if no improvement after 5 epochs
es_callback = EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='min', restore_best_weights=True)


print(f"\n--- Training Baseline MLP for {EPOCHS} Epochs ---")
history_baseline = model_baseline.fit(
    X_train_scaled, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_scaled, Y_val),
    callbacks=[es_callback], # Adding Early Stopping
    verbose=1 # Ensure this is '1' for Appendix 1 screenshots
)

# Re-initialize deep model to ensure fair training if EarlyStopping restored weights
model_deep = create_deep_mlp(INPUT_DIM)

print(f"\n--- Training Deep MLP for {EPOCHS} Epochs ---")
history_deep = model_deep.fit(
    X_train_scaled, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_scaled, Y_val),
    callbacks=[es_callback], # Adding Early Stopping
    verbose=1
)


# Evaluation Function for Reporting Metrics
def evaluate_model_and_report(model, X_data, Y_true, model_name):
    # Predict probabilities on the Test Set
    Y_pred_prob = model.predict(X_data, verbose=0)
    # Convert probabilities to binary class (0 or 1) using 0.5 threshold
    Y_pred_class = (Y_pred_prob > 0.5).astype(int)

    print(f"\n--- Final Results for {model_name} on Test Set (Section 5 Tables) ---")

    # Print Classification Report (Accuracy, Precision, Recall, F1-Score)
    print(classification_report(Y_true, Y_pred_class, target_names=['Ham (0)', 'Spam (1)']))

    # Print Confusion Matrix (for detailed analysis)
    print("Confusion Matrix:")
    print(confusion_matrix(Y_true, Y_pred_class))

    # Get overall accuracy
    acc = accuracy_score(Y_true, Y_pred_class)
    print(f"Overall Test Accuracy: {acc:.4f}")


#Evaluation for both models
evaluate_model_and_report(model_baseline, X_test_scaled, Y_test, "Baseline MLP")
evaluate_model_and_report(model_deep, X_test_scaled, Y_test, "Deep MLP")

Loading Spambase dataset...
Removed 391 duplicate rows. Dataset size: 4210
Data Split: Train=2947, Validation=421, Test=842
Features Normalized using MinMaxScaler (0 to 1).

--- Model 1: Baseline MLP Summary (for Appendix 1) ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         1,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,889 (7.38 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 64)             │         3,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,825 (22.75 KB)

 Trainable params: 5,825 (22.75 KB)

 Non-trainable params: 0 (0.00 B)


--- Training Baseline MLP for 50 Epochs ---
Epoch 1/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.5171 - loss: 0.6892 - val_accuracy: 0.8100 - val_loss: 0.6124
Epoch 2/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8137 - loss: 0.5837 - val_accuracy: 0.8955 - val_loss: 0.4626
Epoch 3/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8752 - loss: 0.4499 - val_accuracy: 0.9002 - val_loss: 0.3625
Epoch 4/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8891 - loss: 0.3618 - val_accuracy: 0.9002 - val_loss: 0.3156
Epoch 5/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8925 - loss: 0.3125 - val_accuracy: 0.9121 - val_loss: 0.2905
Epoch 6/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9064 - loss: 0.2938 - val_accuracy: 0.9097 - val_loss: 0.2797
Epoch 7/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9091 - loss: 0.2762 - val_accuracy: 0.9216 - val_loss: 0.2696
Epoch 8/50
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9088 - lo